In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, roc_curve, auc
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM, Dense, Conv1D, MaxPooling1D, Flatten, GRU, 
    Bidirectional, Attention, Input, LayerNormalization
)
from tensorflow.keras.callbacks import EarlyStopping
import prettytable

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:

physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
else:
    print("No GPU found for TensorFlow")

VISUALIZATION_DIR = "../../visualizations"
MODELS_DIR = "../../models"
os.makedirs(VISUALIZATION_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

def check_gpu_usage():
    print("TensorFlow GPU Available:", tf.test.is_gpu_available())
    if tf.test.is_gpu_available():
        print("TensorFlow Devices:", tf.config.list_physical_devices('GPU'))

def load_and_prepare_data(file_path):
    print("Loading full dataset...")
    df = pd.read_csv(file_path)
    print("Label Distribution:")
    label_counts = df['label'].value_counts()
    print(label_counts)
    print(f"Percentage of Phishing (1): {label_counts.get(1, 0) / len(df) * 100:.2f}%")
    print(f"Percentage of Legitimate (0): {label_counts.get(0, 0) / len(df) * 100:.2f}%")
    
    if not df['label'].isin([0, 1]).all():
        raise ValueError("Labels contain values other than 0 or 1")
    
    feature_columns = [col for col in df.columns if col not in ['url', 'label']]
    X = df[feature_columns].values.astype(np.float16)
    y = df['label'].values
    urls = df['url'].values
    
    scaler = StandardScaler()
    X = scaler.fit_transform(X).astype(np.float16)
    
    X_train, X_test, y_train, y_test, urls_train, urls_test = train_test_split(
        X, y, urls, test_size=0.2, random_state=42, stratify=y
    )
    del df, X, y, urls
    return X_train, X_test, y_train, y_test, urls_train, urls_test, scaler

def create_lstm_model(input_dim):
    model = Sequential([
        LSTM(32, input_shape=(input_dim, 1), return_sequences=True),
        LSTM(16),
        Dense(16, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model, "LSTM Model"

def create_cnn_model(input_dim):
    model = Sequential([
        Conv1D(32, 3, activation='relu', input_shape=(input_dim, 1), padding='same'),
        MaxPooling1D(2),
        Conv1D(16, 3, activation='relu', padding='same'),
        MaxPooling1D(2),
        Flatten(),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model, "CNN Model"

def create_gru_model(input_dim):
    model = Sequential([
        GRU(32, input_shape=(input_dim, 1), return_sequences=True),
        GRU(16),
        Dense(16, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model, "GRU Model"

def create_bilstm_model(input_dim):
    model = Sequential([
        Bidirectional(LSTM(32, return_sequences=True), input_shape=(input_dim, 1)),
        Bidirectional(LSTM(16)),
        Dense(16, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model, "BiLSTM Model"

def create_cnn_gru_model(input_dim):
    model = Sequential([
        Conv1D(32, 3, activation='relu', input_shape=(input_dim, 1), padding='same'),
        MaxPooling1D(2),
        GRU(32, return_sequences=True),
        GRU(16),
        Dense(16, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model, "CNN-GRU Model"

def create_attention_model(input_dim):
    inputs = Input(shape=(input_dim, 1))
    conv = Conv1D(32, 3, activation='relu', padding='same')(inputs)
    attn = Attention()([conv, conv])
    norm = LayerNormalization(epsilon=1e-6)(attn)
    flat = Flatten()(norm)
    dense = Dense(32, activation='relu')(flat)
    outputs = Dense(1, activation='sigmoid')(dense)
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model, "Attention Model"

def calculate_fpr(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    return fpr

def plot_comparison_metrics(histories, model_names, fprs):
    os.makedirs(VISUALIZATION_DIR, exist_ok=True)
    plt.figure(figsize=(12, 8))
    for history, name in zip(histories, model_names):
        plt.plot(history.history['val_accuracy'], label=f'{name} Val Accuracy')
    plt.title('Validation Accuracy Comparison (Deep Learning)')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(VISUALIZATION_DIR, 'val_accuracy_comparison_deep_learning.png'))
    plt.close()

def train_deep_learning_models(features_file, epochs=5):
    print("Checking GPU usage before training (TensorFlow)...")
    check_gpu_usage()
    
    print("Loading and preparing data...")
    X_train, X_test, y_train, y_test, urls_train, urls_test, scaler = load_and_prepare_data(features_file)
    print(f"Data loaded. X_train shape: {X_train.shape}")
    
    X_train_reshaped = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
    X_test_reshaped = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))
    print(f"Data reshaped. X_train_reshaped shape: {X_train_reshaped.shape}")
    
    model_creators = [
        create_lstm_model, create_cnn_model, create_gru_model,
        create_bilstm_model, create_cnn_gru_model, create_attention_model
    ]
    models = []
    model_names = []
    histories = []
    results = []
    
    early_stopping = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
    class_weights = {0: 1.0, 1: len(y_train) / (3 * np.sum(y_train))}
    
    for creator in model_creators:
        model, name = creator(X_train.shape[1])
        print(f"\nTraining {name}")
        if tf.test.is_gpu_available():
            print(f"{name} using GPU: {tf.config.list_physical_devices('GPU')}")
        
        history = model.fit(
            X_train_reshaped, y_train,
            epochs=epochs,
            batch_size=256,
            validation_split=0.2,
            callbacks=[early_stopping],
            class_weight=class_weights,
            verbose=1
        )
        test_loss, test_accuracy = model.evaluate(X_test_reshaped, y_test, verbose=0)
        y_pred = (model.predict(X_test_reshaped, batch_size=256) > 0.5).astype(int)
        models.append(model)
        model_names.append(name)
        histories.append(history)
        results.append({
            'name': name,
            'test_accuracy': test_accuracy,
            'test_loss': test_loss,
            'val_accuracy': history.history['val_accuracy'][-1],
            'fpr': calculate_fpr(y_test, y_pred)
        })
        save_path = os.path.join(MODELS_DIR, f"model_{name.lower().replace(' ', '_')}.h5")
        model.save(save_path)
        print(f"Saved {name} to {save_path}")
        del model
    
    np.save(os.path.join(MODELS_DIR, 'scaler.npy'), scaler)
    
    table = prettytable.PrettyTable()
    table.field_names = ["Model", "Test Accuracy", "Test Loss", "Validation Accuracy", "False Positive Rate"]
    for result in results:
        table.add_row([result['name'], f"{result['test_accuracy']:.4f}", f"{result['test_loss']:.4f}", 
                       f"{result['val_accuracy']:.4f}", f"{result['fpr']:.4f}"])
    print("\nDeep Learning Model Comparison Results:")
    print(table)
    
    plot_comparison_metrics(histories, model_names, [r['fpr'] for r in results])
    
    return models, histories, results

if __name__ == "__main__":
    FEATURES_FILE = "../../data/extracted_features.csv"
    models, histories, results = train_deep_learning_models(FEATURES_FILE, epochs=5)

No GPU found for TensorFlow
Checking GPU usage before training (TensorFlow)...
Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.
TensorFlow GPU Available: False
Loading and preparing data...
Loading full dataset...
Label Distribution:
label
0    1002880
1     285380
Name: count, dtype: int64
Percentage of Phishing (1): 22.15%
Percentage of Legitimate (0): 77.85%
Data loaded. X_train shape: (1030608, 45)
Data reshaped. X_train_reshaped shape: (1030608, 45, 1)


c:\Users\aksha\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



Training LSTM Model
Epoch 1/5
3221/3221 ━━━━━━━━━━━━━━━━━━━━ 185s 56ms/step - accuracy: 0.9844 - loss: 0.0803 - val_accuracy: 0.9982 - val_loss: 0.0058
Epoch 2/5
3221/3221 ━━━━━━━━━━━━━━━━━━━━ 137s 42ms/step - accuracy: 0.9979 - loss: 0.0071 - val_accuracy: 0.9983 - val_loss: 0.0053
Epoch 3/5
3221/3221 ━━━━━━━━━━━━━━━━━━━━ 138s 43ms/step - accuracy: 0.9982 - loss: 0.0051 - val_accuracy: 0.9989 - val_loss: 0.0031
Epoch 4/5
3221/3221 ━━━━━━━━━━━━━━━━━━━━ 141s 44ms/step - accuracy: 0.9986 - loss: 0.0048 - val_accuracy: 0.9990 - val_loss: 0.0028
Epoch 5/5
3221/3221 ━━━━━━━━━━━━━━━━━━━━ 135s 42ms/step - accuracy: 0.9989 - loss: 0.0035 - val_accuracy: 0.9991 - val_loss: 0.0027
1007/1007 ━━━━━━━━━━━━━━━━━━━━ 19s 19ms/step


c:\Users\aksha\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Saved LSTM Model to ../../models\model_lstm_model.h5

Training CNN Model
Epoch 1/5
3221/3221 ━━━━━━━━━━━━━━━━━━━━ 27s 8ms/step - accuracy: 0.9917 - loss: 0.0453 - val_accuracy: 0.9992 - val_loss: 0.0026
Epoch 2/5
3221/3221 ━━━━━━━━━━━━━━━━━━━━ 26s 8ms/step - accuracy: 0.9992 - loss: 0.0031 - val_accuracy: 0.9993 - val_loss: 0.0021
Epoch 3/5
3221/3221 ━━━━━━━━━━━━━━━━━━━━ 26s 8ms/step - accuracy: 0.9993 - loss: 0.0025 - val_accuracy: 0.9994 - val_loss: 0.0019
Epoch 4/5
3221/3221 ━━━━━━━━━━━━━━━━━━━━ 26s 8ms/step - accuracy: 0.9994 - loss: 0.0023 - val_accuracy: 0.9992 - val_loss: 0.0021
Epoch 5/5
3221/3221 ━━━━━━━━━━━━━━━━━━━━ 25s 8ms/step - accuracy: 0.9994 - loss: 0.0019 - val_accuracy: 0.9993 - val_loss: 0.0023
1007/1007 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step


...

Saved GRU Model to ../../models\model_gru_model.h5

Training BiLSTM Model
Epoch 1/5
3221/3221 ━━━━━━━━━━━━━━━━━━━━ 168s 50ms/step - accuracy: 0.9888 - loss: 0.0517 - val_accuracy: 0.9982 - val_loss: 0.0057
Epoch 2/5
3221/3221 ━━━━━━━━━━━━━━━━━━━━ 182s 57ms/step - accuracy: 0.9981 - loss: 0.0063 - val_accuracy: 0.9983 - val_loss: 0.0039
Epoch 3/5
3221/3221 ━━━━━━━━━━━━━━━━━━━━ 168s 52ms/step - accuracy: 0.9983 - loss: 0.0048 - val_accuracy: 0.9987 - val_loss: 0.0036
Epoch 4/5
3221/3221 ━━━━━━━━━━━━━━━━━━━━ 171s 53ms/step - accuracy: 0.9987 - loss: 0.0039 - val_accuracy: 0.9989 - val_loss: 0.0027
Epoch 5/5
3221/3221 ━━━━━━━━━━━━━━━━━━━━ 169s 52ms/step - accuracy: 0.9989 - loss: 0.0032 - val_accuracy: 0.9985 - val_loss: 0.0034
1007/1007 ━━━━━━━━━━━━━━━━━━━━ 23s 22ms/step


c:\Users\aksha\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Saved BiLSTM Model to ../../models\model_bilstm_model.h5

Training CNN-GRU Model
Epoch 1/5
3221/3221 ━━━━━━━━━━━━━━━━━━━━ 91s 27ms/step - accuracy: 0.9856 - loss: 0.0584 - val_accuracy: 0.9991 - val_loss: 0.0029
Epoch 2/5
3221/3221 ━━━━━━━━━━━━━━━━━━━━ 80s 25ms/step - accuracy: 0.9990 - loss: 0.0036 - val_accuracy: 0.9994 - val_loss: 0.0021
Epoch 3/5
3221/3221 ━━━━━━━━━━━━━━━━━━━━ 78s 24ms/step - accuracy: 0.9992 - loss: 0.0029 - val_accuracy: 0.9995 - val_loss: 0.0020
Epoch 4/5
3221/3221 ━━━━━━━━━━━━━━━━━━━━ 78s 24ms/step - accuracy: 0.9994 - loss: 0.0023 - val_accuracy: 0.9995 - val_loss: 0.0016
Epoch 5/5
3221/3221 ━━━━━━━━━━━━━━━━━━━━ 79s 24ms/step - accuracy: 0.9994 - loss: 0.0022 - val_accuracy: 0.9995 - val_loss: 0.0014
1007/1007 ━━━━━━━━━━━━━━━━━━━━ 11s 10ms/step


Saved CNN-GRU Model to ../../models\model_cnn-gru_model.h5

Training Attention Model
Epoch 1/5
3221/3221 ━━━━━━━━━━━━━━━━━━━━ 78s 24ms/step - accuracy: 0.9955 - loss: 0.0131 - val_accuracy: 0.9992 - val_loss: 0.0024
Epoch 2/5
3221/3221 ━━━━━━━━━━━━━━━━━━━━ 75s 23ms/step - accuracy: 0.9990 - loss: 0.0030 - val_accuracy: 0.9986 - val_loss: 0.0043
Epoch 3/5
3221/3221 ━━━━━━━━━━━━━━━━━━━━ 74s 23ms/step - accuracy: 0.9991 - loss: 0.0027 - val_accuracy: 0.9991 - val_loss: 0.0022
Epoch 4/5
3221/3221 ━━━━━━━━━━━━━━━━━━━━ 73s 23ms/step - accuracy: 0.9994 - loss: 0.0022 - val_accuracy: 0.9994 - val_loss: 0.0017
Epoch 5/5
3221/3221 ━━━━━━━━━━━━━━━━━━━━ 73s 23ms/step - accuracy: 0.9994 - loss: 0.0020 - val_accuracy: 0.9995 - val_loss: 0.0015
1007/1007 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step


Saved Attention Model to ../../models\model_attention_model.h5

Deep Learning Model Comparison Results:
+-----------------+---------------+-----------+---------------------+---------------------+
|      Model      | Test Accuracy | Test Loss | Validation Accuracy | False Positive Rate |
+-----------------+---------------+-----------+---------------------+---------------------+
|    LSTM Model   |     0.9990    |   0.0031  |        0.9991       |        0.0012       |
|    CNN Model    |     0.9992    |   0.0020  |        0.9993       |        0.0009       |
|    GRU Model    |     0.9993    |   0.0020  |        0.9993       |        0.0007       |
|   BiLSTM Model  |     0.9989    |   0.0028  |        0.9985       |        0.0011       |
|  CNN-GRU Model  |     0.9995    |   0.0016  |        0.9995       |        0.0005       |
| Attention Model |     0.9995    |   0.0015  |        0.9995       |        0.0005       |
+-----------------+---------------+-----------+---------------------

In [12]:
import sentencepiece
print(sentencepiece.__version__)

ModuleNotFoundError: No module named 'sentencepiece'

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np

VISUALIZATION_DIR = "../../visualizations"
os.makedirs(VISUALIZATION_DIR, exist_ok=True)

models = ['LSTM Model', 'CNN Model', 'GRU Model', 'BiLSTM Model', 'CNN-GRU Model', 'Attention Model']
test_accuracy = [0.9990, 0.9992, 0.9993, 0.9989, 0.9995, 0.9995]
test_loss = [0.0031, 0.0020, 0.0020, 0.0028, 0.0016, 0.0015]
validation_accuracy = [0.9991, 0.9993, 0.9993, 0.9985, 0.9995, 0.9995]
false_positive_rate = [0.0012, 0.0009, 0.0007, 0.0011, 0.0005, 0.0005]

plt.figure(figsize=(10, 6))
plt.bar(models, test_accuracy, color='skyblue')
plt.title('Test Accuracy Comparison (Deep Learning Models)')
plt.xlabel('Model')
plt.ylabel('Test Accuracy')
plt.ylim(0.9985, 1.0000)
plt.xticks(rotation=45, ha='right')
for i, v in enumerate(test_accuracy):
    plt.text(i, v + 0.0001, f'{v:.4f}', ha='center', va='bottom')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(VISUALIZATION_DIR, 'test_accuracy_comparison.png'))
plt.close()

plt.figure(figsize=(10, 6))
plt.bar(models, test_loss, color='lightcoral')
plt.title('Test Loss Comparison (Deep Learning Models)')
plt.xlabel('Model')
plt.ylabel('Test Loss')
plt.ylim(0.0010, 0.0035)  
plt.xticks(rotation=45, ha='right')
for i, v in enumerate(test_loss):
    plt.text(i, v + 0.0001, f'{v:.4f}', ha='center', va='bottom')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(VISUALIZATION_DIR, 'test_loss_comparison.png'))
plt.close()

plt.figure(figsize=(10, 6))
plt.bar(models, validation_accuracy, color='lightgreen')
plt.title('Validation Accuracy Comparison (Deep Learning Models)')
plt.xlabel('Model')
plt.ylabel('Validation Accuracy')
plt.ylim(0.9980, 1.0000)  
plt.xticks(rotation=45, ha='right')
for i, v in enumerate(validation_accuracy):
    plt.text(i, v + 0.0001, f'{v:.4f}', ha='center', va='bottom')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(VISUALIZATION_DIR, 'validation_accuracy_comparison.png'))
plt.close()

plt.figure(figsize=(10, 6))
plt.bar(models, false_positive_rate, color='lightsalmon')
plt.title('False Positive Rate Comparison (Deep Learning Models)')
plt.xlabel('Model')
plt.ylabel('False Positive Rate')
plt.ylim(0.0000, 0.0015)  
plt.xticks(rotation=45, ha='right')
for i, v in enumerate(false_positive_rate):
    plt.text(i, v + 0.0001, f'{v:.4f}', ha='center', va='bottom')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(VISUALIZATION_DIR, 'false_positive_rate_comparison.png'))
plt.close()

print("Plots saved in:", VISUALIZATION_DIR)

Plots saved in: ../../visualizations


In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np

# Directory setup
VISUALIZATION_DIR = "../../visualizations"
os.makedirs(VISUALIZATION_DIR, exist_ok=True)

models = ['LSTM Model', 'CNN Model', 'GRU Model', 'BiLSTM Model', 'CNN-GRU Model', 'Attention Model']
test_accuracy = [0.9990, 0.9992, 0.9993, 0.9989, 0.9995, 0.9995]
test_loss = [0.0031, 0.0020, 0.0020, 0.0028, 0.0016, 0.0015]
validation_accuracy = [0.9991, 0.9993, 0.9993, 0.9985, 0.9995, 0.9995]
false_positive_rate = [0.0012, 0.0009, 0.0007, 0.0011, 0.0005, 0.0005]

plt.figure(figsize=(10, 6))
plt.plot(models, test_accuracy, marker='o', color='skyblue', linestyle='-', linewidth=2, markersize=8)
plt.title('Test Accuracy Comparison')
plt.xlabel('Model')
plt.ylabel('Test Accuracy')
plt.ylim(0.9985, 1.0000) 
plt.xticks(rotation=45, ha='right')
for i, v in enumerate(test_accuracy):
    plt.text(i, v + 0.0001, f'{v:.4f}', ha='center', va='bottom')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(VISUALIZATION_DIR, 'test_accuracy_line_comparison.png'))
plt.close()

plt.figure(figsize=(10, 6))
plt.plot(models, test_loss, marker='o', color='lightcoral', linestyle='-', linewidth=2, markersize=8)
plt.title('Test Loss Comparison (Deep Learning Models)')
plt.xlabel('Model')
plt.ylabel('Test Loss')
plt.ylim(0.0010, 0.0035)
plt.xticks(rotation=45, ha='right')
for i, v in enumerate(test_loss):
    plt.text(i, v + 0.0001, f'{v:.4f}', ha='center', va='bottom')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(VISUALIZATION_DIR, 'test_loss_line_comparison.png'))
plt.close()

plt.figure(figsize=(10, 6))
plt.plot(models, validation_accuracy, marker='o', color='lightgreen', linestyle='-', linewidth=2, markersize=8)
plt.title('Validation Accuracy Comparison (Deep Learning Models)')
plt.xlabel('Model')
plt.ylabel('Validation Accuracy')
plt.ylim(0.9980, 1.0000) 
plt.xticks(rotation=45, ha='right')
for i, v in enumerate(validation_accuracy):
    plt.text(i, v + 0.0001, f'{v:.4f}', ha='center', va='bottom')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(VISUALIZATION_DIR, 'validation_accuracy_line_comparison.png'))
plt.close()

plt.figure(figsize=(10, 6))
plt.plot(models, false_positive_rate, marker='o', color='lightsalmon', linestyle='-', linewidth=2, markersize=8)
plt.title('False Positive Rate Comparison (Deep Learning Models)')
plt.xlabel('Model')
plt.ylabel('False Positive Rate')
plt.ylim(0.0000, 0.0015)  # Zoom in for better visibility
plt.xticks(rotation=45, ha='right')
for i, v in enumerate(false_positive_rate):
    plt.text(i, v + 0.0001, f'{v:.4f}', ha='center', va='bottom')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(VISUALIZATION_DIR, 'false_positive_rate_line_comparison.png'))
plt.close()

print("Line graphs saved in:", VISUALIZATION_DIR)

Line graphs saved in: ../../visualizations
